# 01 - Data acquisition and audit
**Goal:** load the raw public datasets and audit them (shape, types, missing values, duplicates, impossible values).

**Inputs:** `data/raw/spotify_12m/tracks_features.csv` (Kaggle *rodolfofigueroa/spotify-12m-songs*),
`data/raw/spotify_tracks_114k/dataset.csv` (Kaggle *maharshipandya/-spotify-tracks-dataset*).

**Outputs:** `reports/tables/audit_*.csv`, `results.json -> audit`, `data/interim/*.parquet` (typed copies of raw data).

In [1]:
import hashlib
import pandas as pd
import numpy as np
from sonicatlas import DATA, TAB, results

raw = DATA / "raw"
a = pd.read_csv(raw / "spotify_12m" / "tracks_features.csv")
g = pd.read_csv(raw / "spotify_tracks_114k" / "dataset.csv", index_col=0)
print(a.shape, g.shape)

(1204025, 24) (114000, 20)


## Checksums of the raw files (raw data is never edited)

In [2]:
lines = []
for p in sorted(raw.rglob("*.csv")):
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    lines.append(f"{h}  {p.relative_to(DATA).as_posix()}")
(DATA / "CHECKSUMS.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

39ee20762e4bbfe9aefbef7464c5500091eecfdbe0a45d33981898be2530a9e6  raw/spotify_12m/tracks_features.csv
b202fa49909b2d5cef71a04b1d21243cfeb36414535f2ca9272aa646721177bd  raw/spotify_tracks_114k/dataset.csv


## Column types and missing values

In [3]:
def audit(df, name):
    t = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(),
                      "missing_pct": (df.isna().mean() * 100).round(4), "unique": df.nunique()})
    t.to_csv(TAB / f"audit_{name}.csv")
    return t

TAB.mkdir(parents=True, exist_ok=True)
audit(a, "spotify_12m")

,dtype,missing,missing_pct,unique
id,str,0,0.0000,1204025
name,str,3,0.0002,850943
album,str,11,0.0009,106161
album_id,str,0,0.0000,118382
artists,str,0,0.0000,165365
artist_ids,str,0,0.0000,166423
track_number,int64,0,0.0000,50
disc_number,int64,0,0.0000,13
explicit,bool,0,0.0000,2
danceability,float64,0,0.0000,1362


In [4]:
audit(g, "spotify_tracks_114k")

,dtype,missing,missing_pct,unique
track_id,str,0,0.0000,89741
artists,str,1,0.0009,31437
album_name,str,1,0.0009,46589
track_name,str,1,0.0009,73608
popularity,int64,0,0.0000,101
duration_ms,int64,0,0.0000,50697
explicit,bool,0,0.0000,2
danceability,float64,0,0.0000,1174
energy,float64,0,0.0000,2083
key,int64,0,0.0000,12


## Duplicates and impossible values

In [5]:
def norm(s):
    return s.astype(str).str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip()

a_dup_id = int(a["id"].duplicated().sum())
a_key = norm(a["name"]) + "|" + norm(a["artists"])
a_dup_title_artist = int(a_key.duplicated().sum())
g_dup_id = int(g["track_id"].duplicated().sum())
g_key = norm(g["track_name"]) + "|" + norm(g["artists"])
g_dup_title_artist = int(g_key.duplicated().sum())

checks = {
    "tempo_zero": int((a.tempo == 0).sum()),
    "time_signature_zero": int((a.time_signature == 0).sum()),
    "loudness_positive": int((a.loudness > 0).sum()),
    "year_zero_or_before_1900": int((a.year < 1900).sum()),
    "duration_under_30s": int((a.duration_ms < 30_000).sum()),
    "duration_over_20min": int((a.duration_ms > 20 * 60_000).sum()),
}
g_checks = {"tempo_zero": int((g.tempo == 0).sum()), "time_signature_zero": int((g.time_signature == 0).sum()),
            "duration_under_30s": int((g.duration_ms < 30_000).sum()), "popularity_zero": int((g.popularity == 0).sum())}
print(a_dup_id, a_dup_title_artist, g_dup_id, g_dup_title_artist)
print(checks, g_checks)

0 71694 24259 35595
{'tempo_zero': 2777, 'time_signature_zero': 2809, 'loudness_positive': 610, 'year_zero_or_before_1900': 10, 'duration_under_30s': 7675, 'duration_over_20min': 3383} {'tempo_zero': 157, 'time_signature_zero': 163, 'duration_under_30s': 17, 'popularity_zero': 16020}


In [6]:
results.save("audit", {
    "a_rows": len(a), "a_cols": a.shape[1], "g_rows": len(g), "g_cols": g.shape[1],
    "a_missing_total": int(a.isna().sum().sum()),
    "a_missing_by_col": {k: int(v) for k, v in a.isna().sum().items() if v},
    "g_missing_total": int(g.isna().sum().sum()),
    "g_missing_by_col": {k: int(v) for k, v in g.isna().sum().items() if v},
    "a_dup_id": a_dup_id, "a_dup_title_artist": a_dup_title_artist,
    "g_dup_id": g_dup_id, "g_dup_title_artist": g_dup_title_artist,
    "a_checks": checks, "g_checks": g_checks,
    "a_year_min": int(a.loc[a.year >= 1900, "year"].min()), "a_year_max": int(a.year.max()),
    "g_genres": int(g.track_genre.nunique()),
})

## Save typed interim copies (fast Parquet) for the next notebooks

In [7]:
(DATA / "interim").mkdir(exist_ok=True)
a.to_parquet(DATA / "interim" / "spotify_12m.parquet", index=False)
g.to_parquet(DATA / "interim" / "spotify_tracks_114k.parquet", index=False)
print("saved")

saved
